# Verilog Bottom-Up 64-bit · IEEE 754 · ±∞ · NumPy · Torch

§1 Primitives  §2 64-bit Adder  §3 Special Values  §4 FPU Verilog  §5 NumPy Tables  §6 Torch  §7 SymPy

In [ ]:
import sympy as sp
import numpy as np
import torch
import struct, math
from IPython.display import display, Math
sp.init_printing(use_latex='mathjax')
print('ready')

---
# §1 — Primitives: Gates → Half Adder → Full Adder

Bottom-up: smallest verifiable unit first, then compose upward.

In [ ]:
v_half = 'module half_adder(output sum,carry,input a,b); assign sum=a^b; assign carry=a&b; endmodule'
v_full = 'module full_adder(output sum,cout,input a,b,cin); wire s1,c1,c2; half_adder ha1(.sum(s1),.carry(c1),.a(a),.b(b)); half_adder ha2(.sum(sum),.carry(c2),.a(s1),.b(cin)); assign cout=c1|c2; endmodule'
print(v_half); print(v_full)

def half_adder_py(a, b):  return a^b, a&b
def full_adder_py(a, b, cin):
    s1, c1 = half_adder_py(a, b)
    s2, c2 = half_adder_py(s1, cin)
    return s2, c1|c2

print('\nFull adder truth table:'); print(f'{"a":3} {"b":3} {"cin":5} | sum cout')
for a in range(2):
    for b in range(2):
        for cin in range(2):
            s, c = full_adder_py(a, b, cin)
            print(f'{a:3} {b:3} {cin:5}   {s:3} {c}')

---
# §2 — 4-bit CLA → 64-bit Adder

G=a&b (generate), P=a^b (propagate) — all carries in parallel O(log N).

In [ ]:
v_cla4 = ('module cla4(output [3:0] sum,output cout,input [3:0] a,b,input cin);'
           ' wire [3:0] G=a&b,P=a^b; wire [4:0] C;'
           ' assign C[0]=cin;'
           ' assign C[1]=G[0]|(P[0]&C[0]);'
           ' assign C[2]=G[1]|(P[1]&G[0])|(P[1]&P[0]&C[0]);'
           ' assign C[3]=G[2]|(P[2]&G[1])|(P[2]&P[1]&G[0])|(P[2]&P[1]&P[0]&C[0]);'
           ' assign C[4]=G[3]|(P[3]&G[2])|(P[3]&P[2]&G[1])|(P[3]&P[2]&P[1]&G[0])|(P[3]&P[2]&P[1]&P[0]&C[0]);'
           ' assign sum=P^C[3:0]; assign cout=C[4]; endmodule')
print(v_cla4)

def rca_n(a, b, cin, n):
    carry = cin; r = 0
    for i in range(n):
        s, carry = full_adder_py((a>>i)&1, (b>>i)&1, carry)
        r |= s << i
    return r, carry

import random; random.seed(0); errors = 0
for _ in range(10000):
    a = random.randint(0, 2**64-1); b = random.randint(0, 2**64-1)
    ref = (a + b) & ((1<<64)-1)
    got, _ = rca_n(a, b, 0, 64)
    if got != ref: errors += 1
print(f'64-bit RCA 10000 random tests, errors = {errors}')

---
# §3 — IEEE 754 Double: All Special Values

| Field | Bits | +∞ | -∞ | NaN | +0 | denorm |
|-------|------|----|----|-----|----|--------|
| sign  | 1    | 0  | 1  | ×   | 0  | 0      |
| exp   | 11   | 7FF| 7FF| 7FF | 000| 000    |
| mant  | 52   | 0  | 0  | ≠0  | 0  | ≠0     |

In [ ]:
def bits64(v): return int.from_bytes(struct.pack('>d', v), 'big')
def make_fp64(sign, exp, mant):
    b = (sign<<63)|(exp<<52)|(mant&((1<<52)-1))
    return struct.unpack('>d', b.to_bytes(8,'big'))[0]
def classify(v):
    b=bits64(v); s=(b>>63)&1; e=(b>>52)&0x7FF; m=b&((1<<52)-1)
    if e==0x7FF and m==0:  return '+inf' if s==0 else '-inf'
    elif e==0x7FF:          return 'qNaN' if (m>>51)&1 else 'sNaN'
    elif e==0 and m==0:    return '+0'   if s==0 else '-0'
    elif e==0:              return '+denorm' if s==0 else '-denorm'
    else:                   return '+normal' if s==0 else '-normal'

specials = [
    ('+0', 0.0), ('-0', -0.0), ('+inf', math.inf), ('-inf', -math.inf),
    ('qNaN', float('nan')),
    ('+denorm', make_fp64(0,0,1)), ('-denorm', make_fp64(1,0,1)),
    ('min_norm', make_fp64(0,1,0)), ('max_norm', make_fp64(0,0x7FE,(1<<52)-1)),
    ('1.0', 1.0), ('-1.0', -1.0), ('pi', math.pi), ('eps', np.finfo(np.float64).eps),
]
print(f'{"Name":12}  {"Hex":18}  {"Class":10}  Value'); print('-'*70)
for name, val in specials:      # loop over special values
    try: print(f'{name:12}  {bits64(val):016X}    {classify(val):10}  {val}')
    except: print(f'{name:12}  (pack error)')

---
# §4 — FP64 Verilog: Classifier, Special Add, Comparator

In [ ]:
v_cls = ('module fp64_classifier(output reg is_zero,is_inf,is_nan,is_denorm,is_neg,input [63:0] fp);'
         ' wire [10:0] e=fp[62:52]; wire [51:0] m=fp[51:0];'
         ' always @(*) begin is_neg=fp[63]; is_zero=(e==0)&&(m==0);'
         ' is_inf=(e==11h7FF)&&(m==0); is_nan=(e==11h7FF)&&(m!=0); is_denorm=(e==0)&&(m!=0); end endmodule')
print(v_cls)

def fp64_cmp_py(a, b):
    if math.isnan(a) or math.isnan(b): return dict(lt=False,eq=False,gt=False,unord=True)
    return dict(lt=a<b, eq=a==b, gt=a>b, unord=False)

print('\nComparator:')
for a, b in [(math.inf,1.),(-math.inf,1.),(0.,-0.),(float('nan'),1.),(1.,1.),(-1.,1.)]:
    print(f'  {str(a):12} vs {str(b):6} -> {fp64_cmp_py(a,b)}')

---
# §5 — NumPy: Arithmetic Tables + Number Line

In [ ]:
import warnings, matplotlib.pyplot as plt; warnings.filterwarnings('ignore')
OPS = [np.float64(np.inf), np.float64(-np.inf), np.float64(np.nan),
       np.float64(0.), np.float64(-0.), np.float64(1.), np.float64(-1.)]
NMS = ['+inf','-inf','nan','+0','-0','1','-1']

for title, fn in [('Addition', lambda a,b:a+b), ('Multiplication', lambda a,b:a*b)]:
    print(f'\n{title}:'); print(f'{"":8}',end='')
    for n in NMS: print(f'{n:8}',end='')
    print()
    for a,na in zip(OPS,NMS):
        print(f'{na:8}',end='')
        for b in OPS:
            r=fn(a,b)
            if np.isnan(r): rs='NaN'
            elif r==np.inf: rs='+inf'
            elif r==-np.inf: rs='-inf'
            elif r==0: rs='-0' if np.signbit(r) else '+0'
            else: rs=str(r)
            print(f'{rs:8}',end='')
        print()

fig,ax=plt.subplots(figsize=(13,2))
for i,(l,c) in enumerate(zip(['-inf','-norm','-den','+-0','+den','+norm','+inf','NaN'],
    ['tomato','salmon','lightyellow','lightgreen','lightyellow','skyblue','royalblue','lightgray'])):
    ax.add_patch(plt.Rectangle((i*2,0.1),1.8,0.8,color=c,ec='k',lw=1))
    ax.text(i*2+0.9,0.5,l,ha='center',va='center',fontsize=9,fontweight='bold')
ax.set_xlim(-0.1,16.1); ax.set_ylim(0,1); ax.axis('off')
ax.set_title('IEEE 754 float64 number line'); plt.tight_layout(); plt.show()

---
# §6 — Torch: Special Value Propagation at Scale

In [ ]:
torch.set_default_dtype(torch.float64)
N=100000; x=torch.randn(N,dtype=torch.float64)
x[torch.randperm(N)[:100]]=float('inf')
x[torch.randperm(N)[:100]]=float('-inf')
x[torch.randperm(N)[:50]] =float('nan')

for name,fn in [
    ('exp(x)',lambda t:torch.exp(t)), ('sigmoid',lambda t:torch.sigmoid(t)),
    ('tanh',lambda t:torch.tanh(t)), ('relu',lambda t:torch.relu(t)),
    ('1/x',lambda t:1./t), ('x^2',lambda t:t**2),
    ('log|x|+eps',lambda t:torch.log(t.abs()+1e-300)),
    ('clamp(-1,1)',lambda t:t.clamp(-1,1)),
    ('x/(|x|+1e-15)',lambda t:t/(t.abs()+1e-15))
]:                                      # loop over ops
    y=fn(x)
    ni=torch.isinf(y).sum().item(); nn=torch.isnan(y).sum().item()
    print(f'{name:18}  inf={ni:6d}  nan={nn:6d}  normal={N-ni-nn:6d}')

---
# §7 — SymPy: Exact Arithmetic vs float64

In [ ]:
for name,expr in [
    ('1/3',sp.Rational(1,3)), ('sqrt2',sp.sqrt(2)),
    ('pi',sp.pi), ('1/7*7',sp.Rational(1,7)*7),
    ('(sqrt2)^2',sp.sqrt(2)**2), ('sin(pi)',sp.sin(sp.pi)),
    ('Gamma(1/2)',sp.gamma(sp.Rational(1,2))),
    ('oo+1',sp.oo+1), ('oo-oo',sp.oo-sp.oo),
    ('1/oo',1/sp.oo), ('oo*0',sp.oo*0), ('-oo',-sp.oo),
]:                              # loop over exact cases
    try:
        if expr is sp.oo: nv=np.inf
        elif expr == -sp.oo: nv=-np.inf
        elif str(expr) in ('nan','zoo'): nv=np.nan
        else: nv=float(expr.evalf())
        print(f'{name:14}  sympy={str(expr):15}  numpy={nv}')
    except Exception as e:
        print(f'{name:14}  err: {e}')